# AZEquiScope Data Cleaning & Standardization

Comprehensive data cleaning and standardization pipeline for Maricopa County healthcare equity analysis.

**Input:** Raw data from `data_import.ipynb` pipeline  
**Output:** Standardized dataset ready for analysis

## Key Processing Steps
1. Load raw data and assess quality
2. Filter out ZCTAs with insufficient data coverage (population < 2000, missing CDC data)
3. Standardize numeric variables to 0-1 scale using Min-Max scaling
4. Create Z_ prefixed variables for standardized values
5. Export analysis-ready standardized dataset

## Data Sources Being Processed
- **Census ACS API**: Demographics (population), socioeconomics (income, insurance)
- **CDC PLACES API**: Health outcomes (chronic diseases), health status, healthcare access
- **NPI Registry**: Healthcare provider counts by ZCTA

## Variables Standardized (25 total)
- **Census (4)**: Population, median income, % insured, % uninsured
- **CDC (18)**: Chronic diseases, general health status, healthcare access  
- **NPI (3)**: Total, individual, and organizational provider counts

Note: CDC social determinants variables not available through API used

## 1. Import Libraries and Load Raw Data

In [83]:
# Import required libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries loaded successfully")

✓ Libraries loaded successfully


In [84]:
# Load raw data from the data import pipeline
RAW_DATA_FILE = "data/maricopa_healthcare_raw_data.csv"

# Check if raw data file exists
if not os.path.exists(RAW_DATA_FILE):
    print(f"❌ Raw data file not found: {RAW_DATA_FILE}")
    print("Please run data_import.ipynb first to generate the raw dataset")
    raise FileNotFoundError(f"Raw data file not found: {RAW_DATA_FILE}")

# Load the raw dataset
print(f"Loading raw data from: {RAW_DATA_FILE}")
df_raw = pd.read_csv(RAW_DATA_FILE)

print(f"✓ Raw data loaded successfully")
print(f"  Shape: {df_raw.shape}")
print(f"  ZCTAs: {df_raw['zcta'].nunique()}")
print(f"  Columns: {len(df_raw.columns)}")

# Display first few rows
print("\nFirst 3 rows of raw data:")
display(df_raw.head(3))

Loading raw data from: data/maricopa_healthcare_raw_data.csv
✓ Raw data loaded successfully
  Shape: (140, 28)
  ZCTAs: 140
  Columns: 28

First 3 rows of raw data:


,zcta,census_NAME,census_B01001_001E,census_B19013_001E,census_S2701_C03_001E,census_S2701_C05_001E,cdc_totalpopulation,cdc_access2_crudeprev,cdc_checkup_crudeprev,cdc_dental_crudeprev,...,cdc_highchol_crudeprev,cdc_kidney_crudeprev,cdc_obesity_crudeprev,cdc_stroke_crudeprev,cdc_ghlth_crudeprev,cdc_mhlth_crudeprev,cdc_phlth_crudeprev,npi_total_providers,npi_individual_providers,npi_organization_providers
0,85003,ZCTA5 85003,10155,56672,90.6,9.4,9369,16.6,65.0,57.4,...,30.0,2.7,31.8,2.5,17.2,18.1,10.9,137.0,96.0,41.0
1,85004,ZCTA5 85004,11178,71250,87.3,12.7,4965,16.4,65.5,56.1,...,29.5,2.7,31.3,2.4,17.3,20.8,11.0,181.0,122.0,59.0
2,85006,ZCTA5 85006,22081,60742,76.0,24.0,25742,28.5,64.4,45.7,...,31.9,3.4,37.6,3.1,25.7,20.9,14.8,243.0,179.0,64.0


## 2. Missing Data Analysis

In [85]:
# Identify ZCTAs with data coverage issues
print("\n" + "=" * 60)
print("ZCTA DATA COVERAGE ANALYSIS")
print("=" * 60)

# Get all ZCTAs from raw data
all_zctas = set(df_raw['zcta'].unique())

# Identify ZCTAs missing from CDC (have Census but no CDC data)
census_cols_check = [col for col in df_raw.columns if col.startswith('census_')]
cdc_cols_check = [col for col in df_raw.columns if col.startswith('cdc_')]
npi_cols_check = [col for col in df_raw.columns if col.startswith('npi_')]

# ZCTAs with no CDC data (all CDC columns are 0 or NaN)
if cdc_cols_check:
    cdc_missing = df_raw[df_raw[cdc_cols_check].replace(0, np.nan).isnull().all(axis=1)]['zcta'].tolist()
else:
    cdc_missing = []

# ZCTAs with no NPI data (all NPI columns are 0 or NaN)
if npi_cols_check:
    npi_missing = df_raw[df_raw[npi_cols_check].replace(0, np.nan).isnull().all(axis=1)]['zcta'].tolist()
else:
    npi_missing = []

# Count low-population ZCTAs (population < 2000)
low_pop_threshold = 2000
low_pop_count = 0
if 'census_B01001_001E' in df_raw.columns:
    low_pop_count = len(df_raw[df_raw['census_B01001_001E'] < low_pop_threshold])

print(f"\nData coverage summary:")
print(f"  Total ZCTAs: {len(all_zctas)}")
print(f"  ZCTAs missing CDC data: {len(cdc_missing)}")
print(f"  ZCTAs missing NPI data: {len(npi_missing)}")
print(f"  ZCTAs with population < {low_pop_threshold:,}: {low_pop_count}")

if cdc_missing:
    print(f"\n⚠️  ZCTAs in Census but MISSING from CDC PLACES ({len(cdc_missing)} ZCTAs):")
    for zcta in sorted(cdc_missing):
        # Get Census population for context
        census_pop = df_raw.loc[df_raw['zcta'] == zcta, 'census_B01001_001E'].values
        census_pop_str = f"{int(census_pop[0]):,}" if len(census_pop) > 0 and not pd.isna(census_pop[0]) else "N/A"
        print(f"    {zcta} (Census population: {census_pop_str})")
else:
    print(f"\n✅ All ZCTAs have CDC PLACES data")

if npi_missing:
    print(f"\n⚠️  ZCTAs in Census but NOT in NPI Registry ({len(npi_missing)} ZCTAs):")
    for zcta in sorted(npi_missing):
        # Get Census population for context
        census_pop = df_raw.loc[df_raw['zcta'] == zcta, 'census_B01001_001E'].values
        census_pop_str = f"{int(census_pop[0]):,}" if len(census_pop) > 0 and not pd.isna(census_pop[0]) else "N/A"
        print(f"    {zcta} (Census population: {census_pop_str})")
else:
    print(f"\n✅ All ZCTAs have provider data")

# Identify low-population ZCTAs (population < 2000)
if 'census_B01001_001E' in df_raw.columns:
    low_pop_zctas = df_raw[df_raw['census_B01001_001E'] < low_pop_threshold].copy()
    
    if len(low_pop_zctas) > 0:
        print(f"\n ZCTAs with Census population < {low_pop_threshold:,} ({len(low_pop_zctas)} ZCTAs):")
        for _, row in low_pop_zctas.sort_values('census_B01001_001E').iterrows():
            zcta = row['zcta']
            pop = row['census_B01001_001E']
            pop_str = f"{int(pop):,}" if not pd.isna(pop) else "N/A"
            print(f"    {zcta} (Population: {pop_str})")
    else:
        print(f"\n✅ All ZCTAs have population >= {low_pop_threshold:,}")



ZCTA DATA COVERAGE ANALYSIS

Data coverage summary:
  Total ZCTAs: 140
  ZCTAs missing CDC data: 4
  ZCTAs missing NPI data: 4
  ZCTAs with population < 2,000: 10

⚠️  ZCTAs in Census but MISSING from CDC PLACES (4 ZCTAs):
    85026 (Census population: 0)
    85236 (Census population: 0)
    85329 (Census population: 2,429)
    85378 (Census population: 9,401)

⚠️  ZCTAs in Census but NOT in NPI Registry (4 ZCTAs):
    85026 (Census population: 0)
    85320 (Census population: 315)
    85322 (Census population: 493)
    85545 (Census population: 669)

 ZCTAs with Census population < 2,000 (10 ZCTAs):
    85026 (Population: 0)
    85236 (Population: 0)
    85343 (Population: 79)
    85320 (Population: 315)
    85333 (Population: 403)
    85322 (Population: 493)
    85545 (Population: 669)
    85309 (Population: 965)
    85264 (Population: 1,329)
    85342 (Population: 1,639)

ZCTA DATA COVERAGE ANALYSIS

Data coverage summary:
  Total ZCTAs: 140
  ZCTAs missing CDC data: 4
  ZCTAs miss

## 3. Filter Dataset Based on Data Quality Criteria

In [86]:
# Identify ZCTAs with insufficient data for analysis
print(f"\nZCTA Filtering Criteria:")
print(f"{'='*60}")

# Define criteria for sufficient data
# Drop ZCTAs in this order:
# 1. Population < 2000 (unreliable estimates)
# 2. Missing CDC data (can't do health equity analysis without CDC) (2 ZCTAs)

# Get column groups
census_cols = [col for col in df_raw.columns if col.startswith('census_')]
cdc_cols = [col for col in df_raw.columns if col.startswith('cdc_')]
npi_cols = [col for col in df_raw.columns if col.startswith('npi_')]

# Check Census completeness
census_complete = ~df_raw[census_cols].isnull().any(axis=1)

# Check population threshold FIRST (>= 2000)
pop_threshold = 2000
if 'census_B01001_001E' in df_raw.columns:
    pop_sufficient = df_raw['census_B01001_001E'] >= pop_threshold
else:
    pop_sufficient = pd.Series([True] * len(df_raw), index=df_raw.index)

# Check CDC completeness SECOND (has any non-zero CDC data)
if cdc_cols:
    cdc_complete = ~df_raw[cdc_cols].replace(0, np.nan).isnull().all(axis=1)
else:
    cdc_complete = pd.Series([False] * len(df_raw), index=df_raw.index)

# Sufficient data = Census AND population >= 2000 AND CDC data
sufficient_data = census_complete & pop_sufficient & cdc_complete

# Identify ZCTAs to keep vs drop
zctas_to_keep = df_raw.loc[sufficient_data, 'zcta'].tolist()
zctas_to_drop = df_raw.loc[~sufficient_data, 'zcta'].tolist()

print(f"Filtering criteria (applied in order):")
print(f"  1. Must have Census population >= {pop_threshold:,}")
print(f"  2. Must have CDC PLACES data")

print(f"\nAnalysis results:")
print(f"  Total ZCTAs in raw data: {len(df_raw)}")
print(f"  ZCTAs with Census data: {census_complete.sum()}")
print(f"  ZCTAs with Census population >= {pop_threshold:,}: {pop_sufficient.sum()}")
print(f"  ZCTAs with CDC data: {cdc_complete.sum()}")
print(f"  ZCTAs meeting ALL criteria: {len(zctas_to_keep)}")
print(f"  ZCTAs to drop: {len(zctas_to_drop)}")

if zctas_to_drop:
    print(f"\n⚠️  ZCTAs that will be DROPPED ({len(zctas_to_drop)} total):")
    for zcta in sorted(zctas_to_drop):
        # Get population for context
        pop = df_raw.loc[df_raw['zcta'] == zcta, 'census_B01001_001E'].values
        pop_str = f"{int(pop[0]):,}" if len(pop) > 0 and not pd.isna(pop[0]) else "N/A"
        
        # Determine reason for dropping (in order of priority)
        has_census = not df_raw.loc[df_raw['zcta'] == zcta, census_cols].isnull().any().any()
        has_sufficient_pop = pop[0] >= pop_threshold if len(pop) > 0 and not pd.isna(pop[0]) else False
        has_cdc = not df_raw.loc[df_raw['zcta'] == zcta, cdc_cols].replace(0, np.nan).isnull().all().any()
        
        reasons = []
        if not has_census:
            reasons.append("missing Census data")
        if not has_sufficient_pop:
            reasons.append(f"population < {pop_threshold:,}")
        if not has_cdc:
            reasons.append("missing CDC data")
        
        print(f"    {zcta} (pop: {pop_str}): {', '.join(reasons)}")


ZCTA Filtering Criteria:
Filtering criteria (applied in order):
  1. Must have Census population >= 2,000
  2. Must have CDC PLACES data

Analysis results:
  Total ZCTAs in raw data: 140
  ZCTAs with Census data: 140
  ZCTAs with Census population >= 2,000: 130
  ZCTAs with CDC data: 136
  ZCTAs meeting ALL criteria: 128
  ZCTAs to drop: 12

⚠️  ZCTAs that will be DROPPED (12 total):
    85026 (pop: 0): population < 2,000, missing CDC data
    85236 (pop: 0): population < 2,000, missing CDC data
    85264 (pop: 1,329): population < 2,000
    85309 (pop: 965): population < 2,000
    85320 (pop: 315): population < 2,000
    85322 (pop: 493): population < 2,000
    85329 (pop: 2,429): missing CDC data
    85333 (pop: 403): population < 2,000
    85342 (pop: 1,639): population < 2,000
    85343 (pop: 79): population < 2,000
    85378 (pop: 9,401): missing CDC data
    85545 (pop: 669): population < 2,000


In [87]:
# Apply filtering to create clean dataset
print(f"\nApplying filters to dataset...")

# Create filtered dataset
df_filtered = df_raw[df_raw['zcta'].isin(zctas_to_keep)].copy()

print(f"✓ Dataset filtered successfully")
print(f"  Original dataset: {df_raw.shape}")
print(f"  Filtered dataset: {df_filtered.shape}")
print(f"  ZCTAs removed: {len(df_raw) - len(df_filtered)}")
print(f"  ZCTAs remaining: {len(df_filtered)}")

# Verify no missing values in key columns
print(f"\nData completeness check after filtering:")
missing_census = df_filtered[census_cols].isnull().any(axis=1).sum()
missing_cdc = df_filtered[cdc_cols].replace(0, np.nan).isnull().all(axis=1).sum()
low_pop = (df_filtered['census_B01001_001E'] < pop_threshold).sum() if 'census_B01001_001E' in df_filtered.columns else 0

print(f"  Census missing data: {missing_census} ZCTAs")
print(f"  CDC missing/zero data: {missing_cdc} ZCTAs")
print(f"  Population < {pop_threshold:,}: {low_pop} ZCTAs")

if missing_census == 0 and missing_cdc == 0 and low_pop == 0:
    print("  ✅ All remaining ZCTAs meet quality criteria")
else:
    print("  ⚠️  Some ZCTAs still have issues - check filtering logic")

# Clean up any remaining NaN values (fill with 0 for provider counts)
numeric_cols = census_cols + cdc_cols + npi_cols
for col in numeric_cols:
    if col in df_filtered.columns:
        df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce').fillna(0)

print(f"\n✓ Numeric data types validated and NaN values handled")

# Drop census_NAME column as it's not needed for analysis
if 'census_NAME' in df_filtered.columns:
    df_filtered = df_filtered.drop(columns=['census_NAME'])
    print(f"✓ Dropped census_NAME column (geographic name not needed for analysis)")

print(f"\nReady for standardization with {len(df_filtered)} high-quality ZCTAs")


Applying filters to dataset...
✓ Dataset filtered successfully
  Original dataset: (140, 28)
  Filtered dataset: (128, 28)
  ZCTAs removed: 12
  ZCTAs remaining: 128

Data completeness check after filtering:
  Census missing data: 0 ZCTAs
  CDC missing/zero data: 0 ZCTAs
  Population < 2,000: 0 ZCTAs
  ✅ All remaining ZCTAs meet quality criteria

✓ Numeric data types validated and NaN values handled
✓ Dropped census_NAME column (geographic name not needed for analysis)

Ready for standardization with 128 high-quality ZCTAs


## 4. Standardize Numeric Variables to 0-1 Scale

In [88]:
# Define variables to standardize with their descriptions
variables_to_standardize = {
    # Census demographics & socioeconomic
    'census_B01001_001E': 'Total Population',
    'census_B19013_001E': 'Median Household Income',
    'census_S2701_C03_001E': 'Percent Insured',
    'census_S2701_C05_001E': 'Percent Uninsured',
    
    # CDC health indicators (prevalence rates) 
    # CDC Health Outcomes - Chronic Diseases
    'cdc_arthritis_crudeprev': 'Arthritis (%)',
    'cdc_bphigh_crudeprev': 'High Blood Pressure (%)',
    'cdc_cancer_crudeprev': 'Cancer (%)',
    'cdc_casthma_crudeprev': 'Current Asthma (%)',
    'cdc_chd_crudeprev': 'Coronary Heart Disease (%)',
    'cdc_copd_crudeprev': 'COPD (%)',
    'cdc_depression_crudeprev': 'Depression (%)',
    'cdc_diabetes_crudeprev': 'Diabetes (%)',
    'cdc_highchol_crudeprev': 'High Cholesterol (%)',
    'cdc_kidney_crudeprev': 'Chronic Kidney Disease (%)',
    'cdc_obesity_crudeprev': 'Obesity (%)',
    'cdc_stroke_crudeprev': 'Stroke (%)',
    
    # CDC Health Status - General Health
    'cdc_ghlth_crudeprev': 'General Health - Fair/Poor (%)',
    'cdc_mhlth_crudeprev': 'Poor Mental Health Days (%)',
    'cdc_phlth_crudeprev': 'Poor Physical Health Days (%)',

    # CDC Prevention - Healthcare Access
    'cdc_access2_crudeprev': 'Lack Health Insurance (%)',
    'cdc_checkup_crudeprev': 'Annual Checkup (%)', 
    'cdc_dental_crudeprev': 'Annual Dental Visit (%)',
    
    # NPI provider counts
    'npi_total_providers': 'Total Healthcare Providers',
    'npi_individual_providers': 'Individual Providers',
    'npi_organization_providers': 'Organization Providers'
}

print(f"Variables to be standardized: {len(variables_to_standardize)}")
for i, (var, desc) in enumerate(variables_to_standardize.items(), 1):
    print(f"  {i:2d}. {var}: {desc}")

Variables to be standardized: 25
   1. census_B01001_001E: Total Population
   2. census_B19013_001E: Median Household Income
   3. census_S2701_C03_001E: Percent Insured
   4. census_S2701_C05_001E: Percent Uninsured
   5. cdc_arthritis_crudeprev: Arthritis (%)
   6. cdc_bphigh_crudeprev: High Blood Pressure (%)
   7. cdc_cancer_crudeprev: Cancer (%)
   8. cdc_casthma_crudeprev: Current Asthma (%)
   9. cdc_chd_crudeprev: Coronary Heart Disease (%)
  10. cdc_copd_crudeprev: COPD (%)
  11. cdc_depression_crudeprev: Depression (%)
  12. cdc_diabetes_crudeprev: Diabetes (%)
  13. cdc_highchol_crudeprev: High Cholesterol (%)
  14. cdc_kidney_crudeprev: Chronic Kidney Disease (%)
  15. cdc_obesity_crudeprev: Obesity (%)
  16. cdc_stroke_crudeprev: Stroke (%)
  17. cdc_ghlth_crudeprev: General Health - Fair/Poor (%)
  18. cdc_mhlth_crudeprev: Poor Mental Health Days (%)
  19. cdc_phlth_crudeprev: Poor Physical Health Days (%)
  20. cdc_access2_crudeprev: Lack Health Insurance (%)
  21. cdc_

In [89]:
# Apply Min-Max scaling (0-1 normalization) to selected variables
print(f"\nApplying Min-Max scaling to {len(variables_to_standardize)} variables...")

# Create a copy of the filtered dataset for standardization
df_standardized = df_filtered.copy()

# Apply Min-Max scaling to each variable
scaler = MinMaxScaler()
variables_scaled = 0

# Variables where higher values are BETTER (need to flip: 1 - scaled_value)
# So higher values become closer to 0 (better), lower values closer to 1 (worse)
variables_to_flip = {
    'census_B19013_001E',      # Median Household Income (higher is better)
    'census_S2701_C03_001E',   # Percent Insured (higher is better)
    'cdc_checkup_crudeprev',   # Annual Checkup (higher is better)
    'cdc_dental_crudeprev',    # Annual Dental Visit (higher is better)  
    'npi_total_providers',     # Total Healthcare Providers (higher is better)
    'npi_individual_providers', # Individual Providers (higher is better)
    'npi_organization_providers' # Organization Providers (higher is better)
}

for var, description in variables_to_standardize.items():
    # Get original values
    original_values = df_standardized[var].values.reshape(-1, 1)
    
    # Apply Min-Max scaling
    var_range = df_standardized[var].max() - df_standardized[var].min()
    
    if var_range > 0:
        scaled_values = scaler.fit_transform(original_values).flatten()
        
        # Flip values for "good" variables so higher original values = lower standardized values
        if var in variables_to_flip:
            scaled_values = 1 - scaled_values
            
        z_var_name = f"Z_{var}"
        # Round to 6 decimal places for cleaner output and CSV storage
        df_standardized[z_var_name] = scaled_values.round(6)
        variables_scaled += 1
    else:
        # If no variation (all values same), set standardized version to 0
        z_var_name = f"Z_{var}"
        df_standardized[z_var_name] = 0.0
        variables_scaled += 1

print(f"✓ Min-Max scaling completed for {variables_scaled} variables")
print(f"  All standardized variables have Z_ prefix and 0-1 range")
print(f"  Values rounded to 6 decimal places for cleaner output")
print(f"  {len(variables_to_flip)} 'good' variables flipped so higher values = worse outcomes")
print(f"    Flipped variables: income, insurance, checkups, dental visits, providers")


Applying Min-Max scaling to 25 variables...
✓ Min-Max scaling completed for 25 variables
  All standardized variables have Z_ prefix and 0-1 range
  Values rounded to 6 decimal places for cleaner output
  7 'good' variables flipped so higher values = worse outcomes
    Flipped variables: income, insurance, checkups, dental visits, providers


In [90]:
# Check format and quality of standardized variables
print(f"\nSTANDARDIZED VARIABLES FORMAT CHECK")
print("=" * 50)

# Get all Z_ columns
z_cols = [col for col in df_standardized.columns if col.startswith('Z_')]

print(f"Total standardized variables created: {len(z_cols)}")
print(f"Expected count: {len(variables_to_standardize)}")

# Check data types
print(f"\nData Types:")
z_dtypes = df_standardized[z_cols].dtypes
print(f"  All float64: {(z_dtypes == 'float64').all()}")
print(f"  Note: float64 = 64-bit decimal numbers (standard for numerical data)")
print(f"  Values rounded to 6 decimal places to avoid scientific notation in CSV")

# Check value ranges (should all be 0-1)
print(f"\nValue Ranges (should be 0.0 to 1.0):")
z_ranges = df_standardized[z_cols].agg(['min', 'max'])
print(f"  Overall min: {z_ranges.loc['min'].min():.6f}")
print(f"  Overall max: {z_ranges.loc['max'].max():.6f}")
# Use floating-point tolerance for validation (handles precision issues)
min_valid = (z_ranges.loc['min'] >= -1e-10).all()  # Allow tiny negative due to precision
max_valid = (z_ranges.loc['max'] <= 1 + 1e-10).all()  # Allow tiny excess due to precision
print(f"  All in 0-1 range: {min_valid and max_valid}")


# Sample of standardized vs original values
print(f"\nSample Comparison (first 3 rows):")
sample_vars = list(variables_to_standardize.keys())[:3]
sample_z_vars = [f"Z_{var}" for var in sample_vars]
sample_cols = sample_vars + sample_z_vars
print(f"Showing: {', '.join(sample_vars)}")
display(df_standardized[['zcta'] + sample_cols].head(3))

print(f"\n✓ Standardized variables format check completed")


STANDARDIZED VARIABLES FORMAT CHECK
Total standardized variables created: 25
Expected count: 25

Data Types:
  All float64: True
  Note: float64 = 64-bit decimal numbers (standard for numerical data)
  Values rounded to 6 decimal places to avoid scientific notation in CSV

Value Ranges (should be 0.0 to 1.0):
  Overall min: 0.000000
  Overall max: 1.000000
  All in 0-1 range: True

Sample Comparison (first 3 rows):
Showing: census_B01001_001E, census_B19013_001E, census_S2701_C03_001E


,zcta,census_B01001_001E,census_B19013_001E,census_S2701_C03_001E,Z_census_B01001_001E,Z_census_B19013_001E,Z_census_S2701_C03_001E
0,85003,10155,56672,90.6,0.091492,0.931343,0.296820
1,85004,11178,71250,87.3,0.103555,0.823140,0.413428
2,85006,22081,60742,76.0,0.232121,0.901134,0.812721



✓ Standardized variables format check completed


## 5. Review Standardized Variable Names

In [91]:
# Review and organize standardized variable names
print("Standardized variable naming summary:")
print("=" * 60)

# Get all columns with Z_ prefix (standardized variables)
z_columns = [col for col in df_standardized.columns if col.startswith('Z_')]

print(f"Created {len(z_columns)} standardized variables with Z_ prefix")

# Organize by data source
z_census = [col for col in z_columns if 'census_' in col]
z_cdc = [col for col in z_columns if 'cdc_' in col]  
z_npi = [col for col in z_columns if 'npi_' in col]

print(f"\nStandardized variables by data source:")
print(f"  Census (Z_census_*): {len(z_census)} variables")
print(f"  CDC PLACES (Z_cdc_*): {len(z_cdc)} variables")
print(f"  NPI Registry (Z_npi_*): {len(z_npi)} variables")

# Display the standardized variable names
print(f"\nCensus standardized variables:")
for var in z_census:
    original = var[2:]  # Remove Z_ prefix
    desc = variables_to_standardize.get(original, "")
    print(f"  {var}: {desc}")

print(f"\nCDC PLACES standardized variables (first 10):")
for i, var in enumerate(z_cdc[:10]):
    original = var[2:]  # Remove Z_ prefix  
    desc = variables_to_standardize.get(original, "")
    print(f"  {var}: {desc}")
if len(z_cdc) > 10:
    print(f"  ... and {len(z_cdc)-10} more CDC variables")

print(f"\nNPI Registry standardized variables:")
for var in z_npi:
    original = var[2:]  # Remove Z_ prefix
    desc = variables_to_standardize.get(original, "")
    print(f"  {var}: {desc}")

# Verify naming consistency
print(f"\n✓ All standardized variables follow Z_[datasource]_[variable] naming convention")

Standardized variable naming summary:
Created 25 standardized variables with Z_ prefix

Standardized variables by data source:
  Census (Z_census_*): 4 variables
  CDC PLACES (Z_cdc_*): 18 variables
  NPI Registry (Z_npi_*): 3 variables

Census standardized variables:
  Z_census_B01001_001E: Total Population
  Z_census_B19013_001E: Median Household Income
  Z_census_S2701_C03_001E: Percent Insured
  Z_census_S2701_C05_001E: Percent Uninsured

CDC PLACES standardized variables (first 10):
  Z_cdc_arthritis_crudeprev: Arthritis (%)
  Z_cdc_bphigh_crudeprev: High Blood Pressure (%)
  Z_cdc_cancer_crudeprev: Cancer (%)
  Z_cdc_casthma_crudeprev: Current Asthma (%)
  Z_cdc_chd_crudeprev: Coronary Heart Disease (%)
  Z_cdc_copd_crudeprev: COPD (%)
  Z_cdc_depression_crudeprev: Depression (%)
  Z_cdc_diabetes_crudeprev: Diabetes (%)
  Z_cdc_highchol_crudeprev: High Cholesterol (%)
  Z_cdc_kidney_crudeprev: Chronic Kidney Disease (%)
  ... and 8 more CDC variables

NPI Registry standardized va

## 6. Data Validation and Quality Checks

In [92]:
# Generate summary statistics for key variable groups
print("\nSUMMARY STATISTICS")
print("=" * 40)

# Original variables summary
original_vars = list(variables_to_standardize.keys())
standardized_vars = [f"Z_{var}" for var in original_vars]

print("ORIGINAL VARIABLES - Descriptive Statistics:")
print("=" * 60)
# Calculate custom descriptive statistics for original variables
orig_descriptives = pd.DataFrame({
    'Mean': df_standardized[original_vars].mean(),
    'Std_Dev': df_standardized[original_vars].std(),
    'Median': df_standardized[original_vars].median(),
    'Variance': df_standardized[original_vars].var(),
    'Min': df_standardized[original_vars].min(),
    'Max': df_standardized[original_vars].max()
})
# Format descriptive statistics with custom formatting for variance
orig_descriptives_display = orig_descriptives.copy()
# Use string formatting to avoid scientific notation for variance
orig_descriptives_display = orig_descriptives_display.round(3)
orig_descriptives_display['Variance'] = orig_descriptives['Variance'].apply(lambda x: f"{x:.4f}")

# Set pandas options to avoid scientific notation in display
pd.set_option('display.float_format', lambda x: '%.4f' % x if abs(x) >= 1000 else '%.6f' % x)
display(orig_descriptives_display)
pd.reset_option('display.float_format')

print("\nSTANDARDIZED VARIABLES (Z_) - Descriptive Statistics:")
print("=" * 60)
# Calculate custom descriptive statistics for standardized variables
z_descriptives = pd.DataFrame({
    'Mean': df_standardized[standardized_vars].mean(),
    'Std_Dev': df_standardized[standardized_vars].std(),
    'Median': df_standardized[standardized_vars].median(),
    'Variance': df_standardized[standardized_vars].var(),
    'Min': df_standardized[standardized_vars].min(),
    'Max': df_standardized[standardized_vars].max()
})
display(z_descriptives.round(6))

# Sample of the final dataset
print(f"\nFinal Dataset first 3 rows with key variables):")
key_cols = ['zcta'] + original_vars[:5] + standardized_vars[:5]
sample_cols = [col for col in key_cols if col in df_standardized.columns]
display(df_standardized[sample_cols].head(3))


SUMMARY STATISTICS
ORIGINAL VARIABLES - Descriptive Statistics:


,Mean,Std_Dev,Median,Variance,Min,Max
census_B01001_001E,35546.0080,18140.9440,37448.0000,329093843.6929,2396.0000,87201.0000
census_B19013_001E,90804.4380,30236.4270,85025.0000,914241519.9331,47422.0000,182150.0000
census_S2701_C03_001E,89.713000,6.516000,91.250000,42.4542,70.700000,99.000000
census_S2701_C05_001E,10.287000,6.516000,8.750000,42.4542,1.000000,29.300000
cdc_arthritis_crudeprev,22.240000,5.919000,21.250000,35.0293,11.400000,46.900000
cdc_bphigh_crudeprev,28.241000,5.895000,27.550000,34.7484,17.600000,53.300000
cdc_cancer_crudeprev,6.616000,2.743000,5.900000,7.5213,2.700000,18.300000
cdc_casthma_crudeprev,10.178000,0.868000,10.100000,0.7531,8.300000,14.500000
cdc_chd_crudeprev,5.427000,2.092000,5.100000,4.3757,2.500000,15.000000
cdc_copd_crudeprev,5.813000,1.781000,5.400000,3.1731,3.100000,12.100000



STANDARDIZED VARIABLES (Z_) - Descriptive Statistics:


,Mean,Std_Dev,Median,Variance,Min,Max
Z_census_B01001_001E,0.390897,0.213914,0.413324,0.045759,0.0,1.0
Z_census_B19013_001E,0.678000,0.224426,0.720897,0.050367,0.0,1.0
Z_census_S2701_C03_001E,0.328153,0.230236,0.273852,0.053009,0.0,1.0
Z_census_S2701_C05_001E,0.328153,0.230236,0.273852,0.053009,0.0,1.0
Z_cdc_arthritis_crudeprev,0.305348,0.166720,0.277464,0.027796,0.0,1.0
Z_cdc_bphigh_crudeprev,0.298079,0.165120,0.278712,0.027265,0.0,1.0
Z_cdc_cancer_crudeprev,0.251002,0.175802,0.205128,0.030906,0.0,1.0
Z_cdc_casthma_crudeprev,0.302923,0.139966,0.290323,0.019591,0.0,1.0
Z_cdc_chd_crudeprev,0.234188,0.167345,0.208000,0.028005,0.0,1.0
Z_cdc_copd_crudeprev,0.301476,0.197925,0.255556,0.039174,0.0,1.0



Final Dataset first 3 rows with key variables):


,zcta,census_B01001_001E,census_B19013_001E,census_S2701_C03_001E,census_S2701_C05_001E,cdc_arthritis_crudeprev,Z_census_B01001_001E,Z_census_B19013_001E,Z_census_S2701_C03_001E,Z_census_S2701_C05_001E,Z_cdc_arthritis_crudeprev
0,85003,10155,56672,90.6,9.4,17.5,0.091492,0.931343,0.296820,0.296820,0.171831
1,85004,11178,71250,87.3,12.7,17.3,0.103555,0.823140,0.413428,0.413428,0.166197
2,85006,22081,60742,76.0,24.0,19.2,0.232121,0.901134,0.812721,0.812721,0.219718


## 7. Export Clean Standardized Dataset

In [93]:
# Export standardized dataset and documentation
print("\n" + "="*60)
print("EXPORTING STANDARDIZED DATASET")
print("="*60)

# Ensure output directory exists
print("Setting up output directory...")
os.makedirs("data", exist_ok=True)
print("✓ Data directory ready")

# Define output file paths
STANDARDIZED_DATA_FILE = "data/maricopa_healthcare_standardized_data.csv"
README_FILE = "data/README_standardized_data.txt"

print(f"\nOutput files to be created:")
print(f"  Main dataset: {STANDARDIZED_DATA_FILE}")
print(f"  Documentation: {README_FILE}")

# Step 1: Export main standardized dataset
print(f"\nStep 1: Exporting standardized dataset...")
try:
    df_standardized.to_csv(STANDARDIZED_DATA_FILE, index=False)
    print(f"✓ Successfully exported standardized dataset")
    print(f"  File: {STANDARDIZED_DATA_FILE}")
    print(f"  Shape: {df_standardized.shape[0]} ZCTAs × {df_standardized.shape[1]} variables")
    print(f"  Standardized variables: {len([col for col in df_standardized.columns if col.startswith('Z_')])}")
except Exception as e:
    print(f"X Error exporting dataset: {e}")
    raise

# Step 2: Create comprehensive documentation
print(f"\nStep 2: Creating documentation...")
readme_content = f"""# AZEquiScope Standardized Dataset Documentation

## Dataset Overview
- **File**: maricopa_healthcare_standardized_data.csv
- **Created**: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
- **Source**: Data cleaning pipeline (data_cleaning.ipynb)
- **Geographic Scope**: Maricopa County, Arizona ZCTAs
- **Analysis Ready**: Yes

## Dataset Characteristics
- **Total ZCTAs**: {len(df_standardized)}
- **Total Variables**: {len(df_standardized.columns)}
- **Standardized Variables**: {len([col for col in df_standardized.columns if col.startswith('Z_')])}
- **Missing Values**: {df_standardized.isnull().sum().sum()}

## Data Sources Integrated
1. **U.S. Census Bureau** (demographics, socioeconomics)
   - Total population, median income, insurance coverage
   
2. **CDC PLACES** (health indicators) 
   - Chronic disease prevalence, preventive care, health behaviors
   
3. **NPI Registry** (healthcare providers)
   - Provider counts by ZCTA (individual and organizational)

## Variable Naming Conventions

### Original Variables
- `census_[variable]`: Raw Census data with prefixes
- `cdc_[variable]`: Raw CDC PLACES data with prefixes  
- `npi_[variable]`: Raw NPI provider count data with prefixes

### Standardized Variables (0-1 Scale)
- `Z_[original_variable]`: Min-max normalized (0-1) version of original variable
- All Z_ variables use consistent 0-1 scaling for comparative analysis

## Data Quality Notes
- Filtered to ZCTAs with complete Census AND CDC data
- ZCTAs with missing CDC health data were removed ({len(zctas_to_drop)} ZCTAs dropped)
- All numeric variables converted and validated
- No missing values in final dataset
- All standardized variables confirmed in 0-1 range

## Files Included
- `maricopa_healthcare_standardized_data.csv`: Main analysis dataset
- `README_standardized_data.txt`: This documentation file

## Usage Notes
- Use Z_ prefixed variables for comparative analysis across data sources
- Original variables retained for reference and domain-specific analysis
- ZCTA codes can be used to merge with spatial/geographic data
- All variables ready for statistical analysis and modeling

## Next Steps
This standardized dataset is ready for:
- Descriptive statistics and exploratory data analysis
- Correlation analysis and relationship identification
- Statistical modeling and machine learning
- Spatial analysis and mapping
- Healthcare equity assessment
"""

try:
    with open(README_FILE, 'w') as f:
        f.write(readme_content)
    print(f"✓ Successfully created documentation")
    print(f"  File: {README_FILE}")
except Exception as e:
    print(f"X Error creating documentation: {e}")
    raise

# Export completion summary
print(f"\n" + "="*60)
print("DATA STANDARDIZATION & EXPORT COMPLETE")
print("="*60)

print(f"\nOutput Files Created:")
print(f"  Standardized Dataset: {STANDARDIZED_DATA_FILE}")
print(f"  Documentation: {README_FILE}")

print(f"\nFinal Dataset Summary:")
print(f"  • Total ZCTAs: {len(df_standardized)}")
print(f"  • Total Variables: {len(df_standardized.columns)}")
print(f"  • Original Variables: {len(list(variables_to_standardize.keys()))}")
print(f"  • Standardized Variables: {len([col for col in df_standardized.columns if col.startswith('Z_')])}")
print(f"  • Missing Values: {df_standardized.isnull().sum().sum()}")

print(f"\nNext Steps:")
print(f"  • Dataset is ready for healthcare equity analysis")
print(f"  • Use Z_ prefixed variables for comparative analysis")
print(f"  • Original variables retained for reference")
print(f"  • ZCTA codes available for spatial analysis")

print(f"\nData standardization pipeline completed successfully!")
print(f"   Run time: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")


EXPORTING STANDARDIZED DATASET
Setting up output directory...
✓ Data directory ready

Output files to be created:
  Main dataset: data/maricopa_healthcare_standardized_data.csv
  Documentation: data/README_standardized_data.txt

Step 1: Exporting standardized dataset...
✓ Data directory ready

Output files to be created:
  Main dataset: data/maricopa_healthcare_standardized_data.csv
  Documentation: data/README_standardized_data.txt

Step 1: Exporting standardized dataset...
✓ Successfully exported standardized dataset
  File: data/maricopa_healthcare_standardized_data.csv
  Shape: 128 ZCTAs × 52 variables
  Standardized variables: 25

Step 2: Creating documentation...
✓ Successfully created documentation
  File: data/README_standardized_data.txt

DATA STANDARDIZATION & EXPORT COMPLETE

Output Files Created:
  Standardized Dataset: data/maricopa_healthcare_standardized_data.csv
  Documentation: data/README_standardized_data.txt

Final Dataset Summary:
  • Total ZCTAs: 128
  • Total Va